In [0]:
%sql

CREATE DATABASE IF NOT EXISTS bronze_ecommerce;

In [0]:
%python

bronze_df = (
    spark.read
         .option("header","true")
         .option("inferSchema","true")
         .csv("s3://ecommerce-043272859251-eu-north-1-an/ecommerce-transactions-raw.csv")
)

In [0]:
%python
bronze_df.head()

Row(transaction_id='TXN_0000000000', user_id=131958, product_id='PID_193175', product_category='Sports', sub_category='Team Sports', product_price=2998.9126411169977, discount_percent=60, final_price=1199.5650564467992, quantity=4, transaction_date=datetime.date(2022, 8, 17), payment_mode='Wallet', order_status='Returned', delivery_date=datetime.datetime(2022, 8, 20, 0, 0), delivery_days=3.0, return_flag=1, return_reason='Size Mismatch', return_date=datetime.datetime(2022, 8, 30, 0, 0), seller_rating=4.312577996339445, customer_rating=3.7082296130651025, buyer_state='KA', seller_state='MH', customer_segment='Premium', gst_amount=863.6868406416954, shipping_cost=0.0, refund_amount=4798.260225787197, device_type='Desktop', browser_type='Edge', session_duration_min=19.334006017085212, has_reviews=1, loyalty_member=None)

In [0]:
%python
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_ecommerce.bronze_transactions")

In [0]:
%python
from pyspark.sql.functions import col

null_records = bronze_df.filter(
    col("transaction_id").isNull() |
    col("user_id").isNull() |
    col("product_id").isNull()
)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7809169071791447>, line 3
      1 from pyspark.sql.functions import col
----> 3 null_records = bronze_df.filter(
      4     col("transaction_id").isNull() |
      5     col("user_id").isNull() |
      6     col("product_id").isNull()
      7 )

NameError: name 'bronze_df' is not defined

In [0]:
%python
duplicate_records = bronze_df.groupBy(
    "transaction_id"
).count().filter("count > 1")

In [0]:
%python
invalid_price = bronze_df.filter(
    col("product_price") <= 0
)

In [0]:
%python
invalid_qty = bronze_df.filter(
    col("quantity") <= 0
)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7809169071791450>, line 1
----> 1 invalid_qty = bronze_df.filter(
      2     col("quantity") <= 0
      3 )

NameError: name 'bronze_df' is not defined

In [0]:
%python
invalid_delivery = bronze_df.filter(
    col("delivery_date") < col("transaction_date")
)

In [0]:
%python
invalid_price.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_ecommerce.rejected_records")

In [0]:
%sql

CREATE DATABASE IF NOT EXISTS silver_ecommerce;

In [0]:
%python
from pyspark.sql.functions import *

silver_df = (
    bronze_df
    .dropDuplicates(["transaction_id"])
    .filter(col("product_price") > 0)
    .filter(col("quantity") > 0)
)

In [0]:
%python
silver_df = (
    silver_df
    .withColumn(
        "gross_revenue",
        col("final_price") * col("quantity")
    )
)

In [0]:
%python
silver_df = (
    silver_df
    .withColumn(
        "net_revenue",
        col("gross_revenue")
        + coalesce(col("shipping_cost"),lit(0))
        + coalesce(col("gst_amount"),lit(0))
        - coalesce(col("refund_amount"),lit(0))
    )
)

In [0]:
%python
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_ecommerce.silver_transactions")

In [0]:
%sql
SHOW TABLES IN silver_ecommerce;

database,tableName,isTemporary
silver_ecommerce,silver_new_transactions,false
silver_ecommerce,silver_transactions,false


In [0]:
%sql
DESCRIBE silver_ecommerce.silver_transactions;

col_name,data_type,comment
transaction_id,string,null
user_id,int,null
product_id,string,null
product_category,string,null
sub_category,string,null
product_price,double,null
discount_percent,int,null
final_price,double,null
quantity,int,null
transaction_date,date,null


In [0]:
%python
silver_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("silver_ecommerce.silver_new_transactions")

In [0]:
%sql
DESCRIBE silver_ecommerce.silver_new_transactions;

col_name,data_type,comment
transaction_id,string,null
user_id,int,null
product_id,string,null
product_category,string,null
sub_category,string,null
product_price,double,null
discount_percent,int,null
final_price,double,null
quantity,int,null
transaction_date,date,null


In [0]:
%sql
SHOW TABLES IN silver_ecommerce;

database,tableName,isTemporary
silver_ecommerce,silver_new_transactions,false
silver_ecommerce,silver_transactions,false


In [0]:
%sql
MERGE INTO silver_ecommerce.silver_transactions target
USING silver_ecommerce.silver_new_transactions source
ON target.transaction_id = source.transaction_id

WHEN MATCHED THEN
UPDATE SET
target.final_price = source.final_price,
target.quantity = source.quantity,
target.gross_revenue = source.gross_revenue,
target.net_revenue = source.net_revenue,
target.order_status = source.order_status

WHEN NOT MATCHED THEN
INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2000000,2000000,0,0


In [0]:
%sql
SELECT COUNT(*)
FROM silver_ecommerce.silver_transactions;

COUNT(*)
2000000


In [0]:
%python
invalid_delivery = silver_df.filter(
    col("delivery_date") < col("transaction_date")
)

invalid_delivery.count()

32210

In [0]:
%python
silver_df.filter(
    col("gst_amount") < 0
).count()

90718

In [0]:
%python
silver_df.filter(
    col("shipping_cost") < 0
).count()

10003

In [0]:
%python
silver_df.filter(
    (col("customer_rating") < 1) |
    (col("customer_rating") > 5)
).count()

0

In [0]:
%python
silver_df.filter(
    (col("seller_rating") < 1) |
    (col("seller_rating") > 5)
).count()

0

In [0]:
%python
silver_df = (
    silver_df
    .filter(col("transaction_id").isNotNull())
    .filter(col("user_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("delivery_date") >= col("transaction_date"))
    .filter(col("gst_amount") >= 0)
    .filter(col("shipping_cost") >= 0)
    .filter((col("customer_rating") >= 1) & (col("customer_rating") <= 5))
    .filter((col("seller_rating") >= 1) & (col("seller_rating") <= 5))
)

In [0]:
%python
silver_df.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("silver_ecommerce.silver_transactions")

In [0]:
%python
from pyspark.sql import Row

dq_metrics = [
    Row(
        total_records=bronze_df.count(),
        silver_records=silver_df.count(),
        rejected_records=bronze_df.count() - silver_df.count()
    )
]

dq_df = spark.createDataFrame(dq_metrics)

dq_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("bronze_ecommerce.data_quality_metrics")

In [0]:
CREATE DATABASE IF NOT EXISTS gold_ecommerce;

In [0]:
SHOW DATABASES;

databaseName
bronze_ecommerce
default
gold_ecommerce
information_schema
silver_ecommerce


In [0]:
CREATE TABLE IF NOT EXISTS gold_ecommerce.audit_pipeline_runs
(
run_timestamp TIMESTAMP,
records_processed BIGINT,
status STRING
);

In [0]:
INSERT INTO gold_ecommerce.audit_pipeline_runs
SELECT
current_timestamp(),
COUNT(*),
'SUCCESS'
FROM silver_ecommerce.silver_transactions;

num_affected_rows,num_inserted_rows
1,1


In [0]:
drop table  gold_ecommerce.gold_category_sales;

In [0]:
CREATE OR REPLACE TABLE gold_ecommerce.gold_kpis AS
SELECT
    COUNT(DISTINCT transaction_id) AS total_orders,
    COUNT(DISTINCT user_id) AS total_customers,
    SUM(net_revenue) AS total_revenue,
    AVG(net_revenue) AS avg_order_value,
    SUM(refund_amount) AS total_refunds
FROM silver_ecommerce.silver_transactions;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE gold_ecommerce.gold_category_sales AS
SELECT
    product_category,
    COUNT(transaction_id) AS total_orders,
    SUM(quantity) AS total_quantity_sold,
    SUM(net_revenue) AS total_revenue,
    AVG(net_revenue) AS avg_order_value
FROM silver_ecommerce.silver_transactions
GROUP BY product_category;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE gold_ecommerce.gold_monthly_sales AS
SELECT
    DATE_FORMAT(transaction_date, 'yyyy-MM') AS month,
    COUNT(transaction_id) AS total_orders,
    SUM(net_revenue) AS total_revenue
FROM silver_ecommerce.silver_transactions
GROUP BY DATE_FORMAT(transaction_date, 'yyyy-MM')
ORDER BY month;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE gold_ecommerce.gold_payment_analysis AS
SELECT
    payment_mode,
    COUNT(transaction_id) AS total_transactions,
    SUM(net_revenue) AS total_revenue,
    AVG(net_revenue) AS avg_transaction_value
FROM silver_ecommerce.silver_transactions
GROUP BY payment_mode;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE gold_ecommerce.gold_state_sales AS
SELECT
    buyer_state,
    COUNT(transaction_id) AS total_orders,
    SUM(net_revenue) AS total_revenue,
    AVG(net_revenue) AS avg_order_value
FROM silver_ecommerce.silver_transactions
GROUP BY buyer_state;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE gold_ecommerce.gold_customer_analysis AS
SELECT
    user_id,
    COUNT(transaction_id) AS total_orders,
    SUM(net_revenue) AS total_spent,
    AVG(net_revenue) AS avg_order_value,
    MAX(transaction_date) AS last_purchase_date
FROM silver_ecommerce.silver_transactions
GROUP BY user_id;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE gold_ecommerce.gold_return_analysis AS
SELECT
    return_reason,
    COUNT(transaction_id) AS total_returns,
    SUM(refund_amount) AS total_refund_amount
FROM silver_ecommerce.silver_transactions
WHERE refund_amount > 0
GROUP BY return_reason;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE gold_ecommerce.gold_rating_analysis AS
SELECT
    product_category,
    AVG(customer_rating) AS avg_customer_rating,
    AVG(seller_rating) AS avg_seller_rating
FROM silver_ecommerce.silver_transactions
GROUP BY product_category;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE gold_ecommerce.gold_daily_sales AS
SELECT
    transaction_date,
    COUNT(transaction_id) AS total_orders,
    SUM(net_revenue) AS total_revenue
FROM silver_ecommerce.silver_transactions
GROUP BY transaction_date
ORDER BY transaction_date;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE gold_ecommerce.gold_daily_sales AS
SELECT
    transaction_date,
    COUNT(transaction_id) AS total_orders,
    SUM(net_revenue) AS total_revenue
FROM silver_ecommerce.silver_transactions
GROUP BY transaction_date
ORDER BY transaction_date;

num_affected_rows,num_inserted_rows
